# 📒 Notebook 3 — SVD (Matrix Factorization) Model

**Algorithm:** SVD (Singular Value Decomposition) via the `Surprise` library

**How SVD works (simple explanation):**
- Imagine a giant table: rows = users, columns = restaurants, cells = ratings
- Most cells are empty (user hasn't visited that restaurant)
- SVD learns hidden 'taste factors' for each user and restaurant
- It can then predict any missing cell (= predicted rating)
- We sort predicted ratings → top 3 recommendations

**Requires:** Run Notebook 2 first to generate `data/` folder

## 1. Install Dependencies

In [1]:
!pip install scikit-surprise pandas tqdm -q

## 2. Load Data

In [2]:
import pandas as pd
import pickle

train_df      = pd.read_csv("data/ratings_train.csv")
test_df       = pd.read_csv("data/ratings_test.csv")
business_meta = pd.read_csv("data/business_meta.csv")

with open("data/user_encoder.pkl", "rb") as f:
    user_encoder = pickle.load(f)
with open("data/business_encoder.pkl", "rb") as f:
    biz_encoder = pickle.load(f)

print(f"✅ Train: {len(train_df):,}  |  Test: {len(test_df):,}")
print(f"   Unique users     : {train_df['user_id'].nunique():,}")
print(f"   Unique businesses: {train_df['business_id'].nunique():,}")
train_df.head(3)

✅ Train: 1,579,941  |  Test: 394,986
   Unique users     : 77,443
   Unique businesses: 33,016


,user_id,business_id,target_rating
0,6jz_Yr6_AP2WWLbj9gGDpA,gYYMQeg4X8FUcCxXI4c2Tw,5
1,jmQsfhCp-LF4gsBMex1a1Q,m87qQsq2SMJruE9tVQTIrA,5
2,QKg3ZZ-gGccGNKnKIiw9vA,qNBKRYXQikC8TmLVI1Fhgg,5


## 3. Build Surprise Dataset

In [3]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

# Tell Surprise the rating scale
reader = Reader(rating_scale=(1, 5))

# Load train data into Surprise format
train_data = Dataset.load_from_df(
    train_df[["user_id", "business_id", "target_rating"]],
    reader
)
trainset = train_data.build_full_trainset()

print(f"✅ Surprise trainset built")
print(f"   Users     : {trainset.n_users:,}")
print(f"   Items     : {trainset.n_items:,}")
print(f"   Ratings   : {trainset.n_ratings:,}")

✅ Surprise trainset built
   Users     : 77,443
   Items     : 33,016
   Ratings   : 1,579,941


## 4. Train SVD Model

In [4]:
import time

# SVD hyperparameters
# n_factors: number of latent taste dimensions (higher = more expressive, slower)
# n_epochs : training iterations
# lr_all   : learning rate
# reg_all  : regularization (prevents overfitting)
svd_model = SVD(
    n_factors=75,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.02, 
)

print("🏋️ Training SVD model...")
start = time.time()
svd_model.fit(trainset)
elapsed = time.time() - start
print(f"✅ Training complete in {elapsed:.1f}s")

🏋️ Training SVD model...
✅ Training complete in 19.2s


## 5. Evaluate on Test Set

In [5]:
from surprise import accuracy

# Build testset in Surprise format
test_data = Dataset.load_from_df(
    test_df[["user_id", "business_id", "target_rating"]],
    reader
)
# Convert to anti-testset format for predictions
testset = list(zip(
    test_df["user_id"].tolist(),
    test_df["business_id"].tolist(),
    test_df["target_rating"].tolist()
))

predictions = svd_model.test(testset)

rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)

print("=" * 40)
print("SVD Model Evaluation")
print("=" * 40)
print(f"RMSE : {rmse:.4f}  (lower is better, target < 1.0)")
print(f"MAE  : {mae:.4f}   (mean absolute error in stars)")

# Sample predictions vs actual
print("\nSample predictions (actual vs predicted):")
for pred in predictions[:5]:
    print(f"  User: {pred.uid[:8]}... | Biz: {pred.iid[:8]}... | "
          f"Actual: {pred.r_ui} | Predicted: {pred.est:.2f}")

SVD Model Evaluation
RMSE : 1.0742  (lower is better, target < 1.0)
MAE  : 0.8341   (mean absolute error in stars)

Sample predictions (actual vs predicted):
  User: qUF5K_LH... | Biz: aHHK_n_w... | Actual: 5 | Predicted: 4.84
  User: d__nxnRq... | Biz: TOgkopnD... | Actual: 1 | Predicted: 4.01
  User: cF159aEF... | Biz: zewTzLyi... | Actual: 4 | Predicted: 4.34
  User: BKp3iEfF... | Biz: -dzKS5d4... | Actual: 4 | Predicted: 4.48
  User: ixtKgePF... | Biz: w7kKNELh... | Actual: 4 | Predicted: 2.97


## 6. Save SVD Model

In [6]:
# ── Save SVD factors (no scikit-surprise needed to load) ──────────────────
import pickle, os
os.makedirs("models", exist_ok=True)

# Extract raw numpy arrays from the trained Surprise SVD object
svd_factors = {
    "pu":          svd_model.pu,                    # user latent factors  (n_users  × n_factors)
    "qi":          svd_model.qi,                    # item latent factors  (n_items  × n_factors)
    "bu":          svd_model.bu,                    # user biases          (n_users,)
    "bi":          svd_model.bi,                    # item biases          (n_items,)
    "global_mean": svd_model.trainset.global_mean,  # scalar global mean
    # store the inner-id → string-id mappings so we can look up by user/biz string ID
    "user_inner2id": {v: k for k, v in svd_model.trainset._raw2inner_id_users.items()},
    "user_id2inner": dict(svd_model.trainset._raw2inner_id_users),
    "item_inner2id": {v: k for k, v in svd_model.trainset._raw2inner_id_items.items()},
    "item_id2inner": dict(svd_model.trainset._raw2inner_id_items),
}

with open("models/svd_factors.pkl", "wb") as f:
    pickle.dump(svd_factors, f)

print("✅ SVD factors saved to models/svd_factors.pkl")
print(f"   pu shape : {svd_factors['pu'].shape}")
print(f"   qi shape : {svd_factors['qi'].shape}")
print(f"   Users    : {len(svd_factors['user_id2inner']):,}")
print(f"   Items    : {len(svd_factors['item_id2inner']):,}")

✅ SVD factors saved to models/svd_factors.pkl
   pu shape : (77443, 75)
   qi shape : (33016, 75)
   Users    : 77,443
   Items    : 33,016


## 7. Test: Get Top-3 Recommendations for an Existing User

In [7]:
def svd_recommend(user_id, city=None, state=None, top_k=3):
    """
    Get top-K restaurant recommendations for an existing user using SVD.
    
    Args:
        user_id : The user's string ID from the database
        city    : Optional city filter (e.g. 'Philadelphia')
        state   : Optional state filter (e.g. 'PA')
        top_k   : Number of recommendations to return
    """
    # Check if user exists in training data
    if user_id not in user_encoder["str2idx"]:
        print(f"⚠️  User '{user_id}' not found in training data. Use Notebook 5 for new users.")
        return []

    # Get all businesses the user has already rated
    rated_biz = set(train_df[train_df["user_id"] == user_id]["business_id"].tolist())

    # Filter candidate businesses by location if specified
    candidates = business_meta.copy()
    if city:
        candidates = candidates[candidates["city"].str.lower() == city.lower()]
    if state:
        candidates = candidates[candidates["state"].str.upper() == state.upper()]

    # Remove already-rated businesses
    candidates = candidates[~candidates["business_id"].isin(rated_biz)]

    if candidates.empty:
        print(f"⚠️  No unrated businesses found for city={city}, state={state}")
        return []

    # Predict ratings for all candidates
    preds = []
    for _, biz in candidates.iterrows():
        pred = svd_model.predict(user_id, biz["business_id"])
        preds.append({
            "business_id":   biz["business_id"],
            "business_name": biz["business_name"],
            "city":          biz["city"],
            "state":         biz["state"],
            "avg_stars":     biz["business_avg_stars"],
            "categories":    biz["categories"],
            "svd_score":     pred.est
        })

    # Sort by predicted rating, return top K
    preds_df = pd.DataFrame(preds).sort_values("svd_score", ascending=False).head(top_k)

    print(f"\n🎯 SVD Top-{top_k} for user '{user_id[:12]}...' in {city or 'any city'}, {state or 'any state'}")
    print("─" * 65)
    for i, row in preds_df.iterrows():
        print(f"#{i+1} {row['business_name']} ({row['city']}, {row['state']})")
        print(f"   Categories  : {row['categories']}")
        print(f"   Avg Stars   : ⭐{row['avg_stars']}")
        print(f"   SVD Score   : {row['svd_score']:.3f}")
    return preds_df


# Test with a real user from the training data
sample_user = train_df["user_id"].iloc[0]
print(f"Testing with user: {sample_user}")
results = svd_recommend(sample_user, top_k=3)

Testing with user: 6jz_Yr6_AP2WWLbj9gGDpA

🎯 SVD Top-3 for user '6jz_Yr6_AP2W...' in any city, any state
─────────────────────────────────────────────────────────────────
#30271 Cafe Roma Bakery (Philadelphia, PA)
   Categories  : Restaurants, Food, Bakeries, Donuts
   Avg Stars   : ⭐4.5
   SVD Score   : 5.000
#12748 Artisan Boulanger Patissier (Philadelphia, PA)
   Categories  : Food, Restaurants, Bakeries
   Avg Stars   : ⭐4.5
   SVD Score   : 4.998
#15485 Lazzaroli Pasta (Nashville, TN)
   Categories  : Restaurants, Food, Pasta Shops, Ethnic Food, Italian, Specialty Food, Grocery, International Grocery
   Avg Stars   : ⭐5.0
   SVD Score   : 4.955


## 8. Test with City/State Filter

In [8]:
# Try with location filter
sample_user = train_df["user_id"].iloc[0]

# Find a city that exists in the data
sample_city  = business_meta["city"].value_counts().index[0]
sample_state = business_meta[business_meta["city"] == sample_city]["state"].iloc[0]
print(f"Testing city filter: {sample_city}, {sample_state}")

svd_recommend(sample_user, city=sample_city, state=sample_state, top_k=3)

Testing city filter: Philadelphia, PA

🎯 SVD Top-3 for user '6jz_Yr6_AP2W...' in Philadelphia, PA
─────────────────────────────────────────────────────────────────
#3698 Cafe Roma Bakery (Philadelphia, PA)
   Categories  : Restaurants, Food, Bakeries, Donuts
   Avg Stars   : ⭐4.5
   SVD Score   : 5.000
#1706 Artisan Boulanger Patissier (Philadelphia, PA)
   Categories  : Food, Restaurants, Bakeries
   Avg Stars   : ⭐4.5
   SVD Score   : 4.998
#755 Cafe Mi Quang (Philadelphia, PA)
   Categories  : Restaurants, Vietnamese
   Avg Stars   : ⭐5.0
   SVD Score   : 4.844


,business_id,business_name,city,state,avg_stars,categories,svd_score
3697,JfKe2LI_1F8NhqOsKzHggg,Cafe Roma Bakery,Philadelphia,PA,4.5,"Restaurants, Food, Bakeries, Donuts",5.000000
1705,RFo876v_A63N2JfOTq1p-g,Artisan Boulanger Patissier,Philadelphia,PA,4.5,"Food, Restaurants, Bakeries",4.997848
754,fq1yCVBgBB7s6V-D68NO1g,Cafe Mi Quang,Philadelphia,PA,5.0,"Restaurants, Vietnamese",4.843522
